# BYOL Downstream Classification Evaluation

Evaluates how well the BYOL encoder transfers to downstream classification tasks using the catalogue system.

Sections:
- **0** Configuration
- **1** Imports & catalogue summary
- **2** Load encoder
- **3** Extract / load projections
- **4** Linear probe (all labelled datasets)
- **5** Fine-tuning evaluation (MiraBest & RadioGalaxyDataset)
- **6** Label-fraction experiment (LoTSS classical_pure)
- **7** Cross-dataset generalisation
- **8** Summary table


## 0. Configuration

In [ ]:
CONFIG = {
    "checkpoint": "outputs/run_cnxt_pca_pond_step_wd_20260424_1125/byol_model_best.pt",
    "catalogue":  "catalogues/multi_all.yaml",
    "colour_by":  "dataset",   # "dataset" | "label"
    "force":      False,        # re-extract projections even if cached
    "root":       ".",
}
OUT_DIR = "notebooks/outputs"
SEED    = 42


## 1. Imports & Setup

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Add project src and scripts to path
_root = os.path.abspath("..")
for _p in [os.path.join(_root, "src"), os.path.join(_root, "scripts")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from suplat.data.catalogue import Catalogue
from embed_and_umap import load_encoder, extract_from_array

os.makedirs(OUT_DIR, exist_ok=True)
print("Imports OK")


### 1.1 Materialise catalogue & print summary

In [ ]:
mat    = Catalogue.from_yaml(CONFIG["catalogue"]).materialise(root=CONFIG["root"])
splits = mat.get_split_datasets()

rows = []
for entry in mat._entries:
    name = entry.dataset
    if name in mat._labelled:
        data = mat._labelled[name]
        sp   = mat._splits[name]
        rows.append({
            "dataset":     name,
            "n_train":     len(sp["train"]),
            "n_val":       len(sp["val"]),
            "n_test":      len(sp["test"]),
            "label_names": ", ".join(data["label_names"][:5])
                           + ("\u2026" if len(data["label_names"]) > 5 else ""),
            "f_labels":    entry.f_labels,
            "type":        "labelled",
        })
    else:
        rows.append({
            "dataset":     name,
            "n_train":     len(mat._unlabelled[name]),
            "n_val":       0,
            "n_test":      0,
            "label_names": "\u2014",
            "f_labels":    entry.f_labels,
            "type":        "unlabelled",
        })

summary_df = pd.DataFrame(rows).set_index("dataset")
print(f"Catalogue: {CONFIG['catalogue']}")
display(summary_df)


## 2. Load Encoder

Loads the BYOL checkpoint and returns the online encoder + projector. Auto-detects model type from the checkpoint config.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

encoder, projector = load_encoder(CONFIG["checkpoint"], device)
encoder.eval()
projector.eval()

# Detect dimensions via dummy forward pass
with torch.no_grad():
    _dummy   = torch.zeros(1, 1, 89, 89).to(device)
    enc_dim  = encoder(_dummy).shape[-1]
    proj_dim = projector(encoder(_dummy)).shape[-1]

print(f"\nEncoder output dim  : {enc_dim}")
print(f"Projector output dim: {proj_dim}")


## 3. Extract / Load Projections

For each labelled dataset, pass all images (train + val + test combined) through
`encoder → projector` and cache the results. Re-run with `CONFIG["force"]=True`
to overwrite the cache.

Saved files per dataset (in `<checkpoint_dir>/projections/`):
- `{name}_projections.npy` — shape `(N, proj_dim)`
- `{name}_labels.npy`      — shape `(N,)` int64, 1-D label per image
- `{name}_splits.npy`      — shape `(N,)` int64, 0=train / 1=val / 2=test


In [ ]:
checkpoint_dir = os.path.dirname(os.path.abspath(CONFIG["checkpoint"]))
proj_dir       = os.path.join(checkpoint_dir, "projections")
os.makedirs(proj_dir, exist_ok=True)
print(f"Projection cache dir: {proj_dir}")


def load_or_extract_from_views(name, split_views_dict, tag=None):
    """Extract projections for all images in a labelled dataset across all splits.

    Parameters
    ----------
    name            : dataset name (e.g. "mirabest")
    split_views_dict: {"train": [SplitView], "val": [...], "test": [...]}
    tag             : optional suffix for cache filename (e.g. "classical_pure")

    Returns (projs, labels_1d, split_ids).
    """
    cache_name = f"{name}_{tag}" if tag else name
    proj_path  = os.path.join(proj_dir, f"{cache_name}_projections.npy")
    lbl_path   = os.path.join(proj_dir, f"{cache_name}_labels.npy")
    split_path = os.path.join(proj_dir, f"{cache_name}_splits.npy")

    if not CONFIG["force"] and os.path.exists(proj_path):
        n = np.load(proj_path, mmap_mode="r").shape[0]
        print(f"  {cache_name}: {n} projections loaded from cache")
        return np.load(proj_path), np.load(lbl_path), np.load(split_path)

    all_imgs, all_lbls, all_split_ids = [], [], []
    for split_name, code_id in [("train", 0), ("val", 1), ("test", 2)]:
        for sv in split_views_dict.get(split_name, []):
            if sv.dataset != name:
                continue
            all_imgs.append(sv.images)
            lbls    = sv.labels
            lbls_1d = (lbls.argmax(axis=1) if lbls.ndim > 1 else lbls).astype(np.int64)
            all_lbls.append(lbls_1d)
            all_split_ids.append(np.full(len(sv.images), code_id, dtype=np.int64))

    images_all = np.concatenate(all_imgs)
    lbls_all   = np.concatenate(all_lbls)
    split_ids  = np.concatenate(all_split_ids)

    projs = extract_from_array(encoder, projector, images_all, device)
    np.save(proj_path,  projs)
    np.save(lbl_path,   lbls_all)
    np.save(split_path, split_ids)
    print(f"  {cache_name}: {len(projs)} projections extracted → {proj_dir}/")
    return projs, lbls_all, split_ids


In [ ]:
projs_cache = {}  # name -> {projs, labels, split_ids}

print("=== multi_all labelled datasets ===")
labelled_names = sorted({sv.dataset for svs in splits.values() for sv in svs})
for name in labelled_names:
    p, l, s = load_or_extract_from_views(name, splits)
    projs_cache[name] = {"projs": p, "labels": l, "split_ids": s}

print("\n=== LoTSS classical_pure (separate catalogue) ===")
mat_cp    = Catalogue.from_yaml("catalogues/lotss_classical_pure.yaml").materialise(root=CONFIG["root"])
splits_cp = mat_cp.get_split_datasets()
p, l, s   = load_or_extract_from_views("lotss", splits_cp, tag="classical_pure")
projs_cache["lotss_classical_pure"] = {"projs": p, "labels": l, "split_ids": s}

print("\nProjection shapes:")
for k, v in projs_cache.items():
    sids = v["split_ids"]
    print(f"  {k:<28} {str(v['projs'].shape):<16}  "
          f"train={(sids==0).sum()}  val={(sids==1).sum()}  test={(sids==2).sum()}")


## 4. Linear Probe Evaluation

Fit a logistic regression (sklearn, `max_iter=1000`, `C=1.0`, `random_state=42`) on
train-split projections. Projections are `StandardScaler`-normalised (fit on train only).

Datasets:
- **MiraBest** — binary FRI / FRII
- **RadioGalaxyDataset** — 4 classes (FRI, FRII, Compact, Bent)
- **LoTSS classical_pure** — binary FRI / FRII (unambiguous sources only)


In [ ]:
def run_linear_probe(cache_entry, n_classes):
    """Fit logistic regression on train projections, report val & test metrics.

    Returns dict with keys "val", "test" (each: accuracy/f1/auc/n),
    plus "clf" and "scaler" for reuse in Section 7.
    """
    projs     = cache_entry["projs"]
    labels    = cache_entry["labels"]
    split_ids = cache_entry["split_ids"]

    X = {s: projs[split_ids == i]  for s, i in [("train",0),("val",1),("test",2)]}
    y = {s: labels[split_ids == i] for s, i in [("train",0),("val",1),("test",2)]}

    scaler     = StandardScaler()
    X["train"] = scaler.fit_transform(X["train"])
    X["val"]   = scaler.transform(X["val"])
    X["test"]  = scaler.transform(X["test"])

    clf = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
    clf.fit(X["train"], y["train"])

    results = {}
    for split in ("val", "test"):
        y_pred = clf.predict(X[split])
        y_prob = clf.predict_proba(X[split])
        acc    = accuracy_score(y[split], y_pred)
        f1     = f1_score(y[split], y_pred, average="macro", zero_division=0)
        if n_classes == 2:
            auc = roc_auc_score(y[split], y_prob[:, 1])
        else:
            auc = roc_auc_score(
                label_binarize(y[split], classes=list(range(n_classes))),
                y_prob, multi_class="ovr", average="macro",
            )
        results[split] = {"accuracy": acc, "f1": f1, "auc": auc, "n": len(y[split])}

    results["clf"]    = clf
    results["scaler"] = scaler
    return results


In [ ]:
# Reload from cache if running this section in a fresh kernel
def _ensure(name, tag=None):
    key = f"{name}_{tag}" if tag else name
    if key not in projs_cache:
        cn = f"{name}_{tag}" if tag else name
        projs_cache[key] = {
            "projs":     np.load(os.path.join(proj_dir, f"{cn}_projections.npy")),
            "labels":    np.load(os.path.join(proj_dir, f"{cn}_labels.npy")),
            "split_ids": np.load(os.path.join(proj_dir, f"{cn}_splits.npy")),
        }

_ensure("mirabest")
_ensure("radio_galaxy_dataset")
_ensure("lotss", tag="classical_pure")

probe_results = {}
probe_results["mirabest"]             = run_linear_probe(projs_cache["mirabest"],             n_classes=2)
probe_results["radio_galaxy_dataset"] = run_linear_probe(projs_cache["radio_galaxy_dataset"], n_classes=4)
probe_results["lotss_classical_pure"] = run_linear_probe(projs_cache["lotss_classical_pure"], n_classes=2)

rows = []
for ds, res in probe_results.items():
    r = res["test"]
    rows.append({"dataset": ds, "n_test": r["n"],
                 "accuracy": round(r["accuracy"], 3),
                 "macro_F1": round(r["f1"], 3),
                 "macro_AUC": round(r["auc"], 3)})
df_probe = pd.DataFrame(rows).set_index("dataset")
print("Linear probe — test-split results:")
display(df_probe)


## 5. Fine-tuning Evaluation (MiraBest & RadioGalaxyDataset)

Attach a single `Linear(enc_dim, n_classes)` head to the **frozen** encoder
(encoder output, not projector output) and train with Adam (lr=1e-4) for up to
20 epochs with early stopping (patience=5) on val loss.


In [ ]:
def run_finetune(name, split_views_dict, n_classes,
                 n_epochs=20, lr=1e-4, patience=5, batch_size=32):
    """Train a Linear head on frozen encoder outputs.

    Returns dict with accuracy / f1 / auc on the test split.
    """
    def get_split(sname):
        views = [sv for sv in split_views_dict[sname] if sv.dataset == name]
        if not views:
            return np.zeros((0, 89, 89), np.float32), np.zeros(0, np.int64)
        imgs = np.concatenate([v.images for v in views]).astype(np.float32)
        lbls = np.concatenate([
            v.labels.astype(np.int64) if v.labels.ndim == 1
            else v.labels.argmax(axis=1).astype(np.int64)
            for v in views
        ])
        return imgs, lbls

    tr_imgs, tr_lbls = get_split("train")
    va_imgs, va_lbls = get_split("val")
    te_imgs, te_lbls = get_split("test")
    print(f"  {name}: train={len(tr_imgs)}, val={len(va_imgs)}, test={len(te_imgs)}")

    # Freeze encoder
    for p in encoder.parameters():
        p.requires_grad = False
    encoder.eval()

    head      = nn.Linear(enc_dim, n_classes).to(device)
    nn.init.xavier_uniform_(head.weight)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    rng       = np.random.default_rng(SEED)

    @torch.no_grad()
    def get_enc(imgs_np):
        return encoder(torch.from_numpy(imgs_np[:, None]).to(device))

    best_val_loss, best_state, wait = float("inf"), None, 0

    for epoch in range(n_epochs):
        head.train()
        for i_start in range(0, len(tr_imgs), batch_size):
            b     = rng.permutation(len(tr_imgs))[i_start:i_start + batch_size]
            z     = get_enc(tr_imgs[b])
            y_b   = torch.from_numpy(tr_lbls[b]).to(device)
            loss  = criterion(head(z), y_b)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

        head.eval()
        with torch.no_grad():
            val_loss = criterion(head(get_enc(va_imgs)),
                                 torch.from_numpy(va_lbls).to(device)).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.clone() for k, v in head.state_dict().items()}
            wait          = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"    Early stop at epoch {epoch+1}  "
                      f"(best val loss {best_val_loss:.4f})")
                break

    head.load_state_dict(best_state)
    head.eval()
    with torch.no_grad():
        logits = head(get_enc(te_imgs))
        y_prob = torch.softmax(logits, dim=1).cpu().numpy()
        y_pred = logits.argmax(dim=1).cpu().numpy()

    acc = accuracy_score(te_lbls, y_pred)
    f1  = f1_score(te_lbls, y_pred, average="macro", zero_division=0)
    if n_classes == 2:
        auc = roc_auc_score(te_lbls, y_prob[:, 1])
    else:
        auc = roc_auc_score(
            label_binarize(te_lbls, classes=list(range(n_classes))),
            y_prob, multi_class="ovr", average="macro",
        )

    # Restore encoder gradients
    for p in encoder.parameters():
        p.requires_grad = True

    return {"accuracy": acc, "f1": f1, "auc": auc, "n_test": len(te_lbls)}


In [ ]:
print("Fine-tuning MiraBest...")
ft_mirabest = run_finetune("mirabest", splits, n_classes=2)

print("\nFine-tuning RadioGalaxyDataset...")
ft_rgd = run_finetune("radio_galaxy_dataset", splits, n_classes=4)

ft_results = {"mirabest": ft_mirabest, "radio_galaxy_dataset": ft_rgd}
print("\nFine-tune — test-split results:")
for ds, r in ft_results.items():
    print(f"  {ds:<28}  acc={r['accuracy']:.3f}  "
          f"F1={r['f1']:.3f}  AUC={r['auc']:.3f}")


In [ ]:
# Grouped bar chart: linear probe vs fine-tune per dataset x metric
datasets_ft  = ["mirabest", "radio_galaxy_dataset"]
ds_labels_ft = ["MiraBest", "RadioGalaxyDataset"]
metrics      = ["accuracy", "f1", "auc"]
m_labels     = ["Accuracy", "Macro F1", "Macro AUC"]
clr          = {"lp": "#4C72B0", "ft": "#C44E52"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
x = np.arange(len(metrics))
w = 0.35

for ax, ds, dsl in zip(axes, datasets_ft, ds_labels_ft):
    lp_vals = [probe_results[ds]["test"][m] for m in metrics]
    ft_vals = [ft_results[ds][m] for m in metrics]

    b1 = ax.bar(x - w/2, lp_vals, w, label="Linear probe (projections)", color=clr["lp"])
    b2 = ax.bar(x + w/2, ft_vals, w, label="Fine-tune head (encodings)",  color=clr["ft"])

    for bars in (b1, b2):
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.01,
                    f"{bar.get_height():.2f}",
                    ha="center", va="bottom", fontsize=8)

    ax.set_title(dsl, fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(m_labels)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Score")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Linear Probe (projections) vs Fine-tune Head (encodings)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/probe_vs_finetune.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {OUT_DIR}/probe_vs_finetune.png")


## 6. Label-Fraction Experiment (LoTSS classical_pure)

Train logistic regression probes on subsampled fractions of the LoTSS
classical_pure **train-split** projections. Test accuracy and macro-F1 are
evaluated on the full fixed test split.


In [ ]:
_ensure("lotss", tag="classical_pure")

cp         = projs_cache["lotss_classical_pure"]
X_tr_all   = cp["projs"][cp["split_ids"] == 0]
y_tr_all   = cp["labels"][cp["split_ids"] == 0]
X_te       = cp["projs"][cp["split_ids"] == 2]
y_te       = cp["labels"][cp["split_ids"] == 2]

scaler_frac  = StandardScaler().fit(X_tr_all)
X_tr_all_s   = scaler_frac.transform(X_tr_all)
X_te_s       = scaler_frac.transform(X_te)

fractions    = [0.05, 0.10, 0.25, 0.50, 0.75, 1.0]
frac_results = {}
rng_fr       = np.random.default_rng(SEED)

for f in fractions:
    n   = max(4, int(f * len(X_tr_all_s)))   # >=2 per class for binary
    idx = rng_fr.choice(len(X_tr_all_s), size=n, replace=False)
    clf = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
    clf.fit(X_tr_all_s[idx], y_tr_all[idx])
    y_pred = clf.predict(X_te_s)
    frac_results[f] = {
        "n_train":  n,
        "accuracy": accuracy_score(y_te, y_pred),
        "f1":       f1_score(y_te, y_pred, average="macro", zero_division=0),
    }
    print(f"  f={f:.2f}  n={n:4d}  "
          f"acc={frac_results[f]['accuracy']:.3f}  "
          f"F1={frac_results[f]['f1']:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
xs_pct = [f * 100 for f in fractions]

for ax, metric, ylabel in zip(axes, ["accuracy", "f1"], ["Accuracy", "Macro F1"]):
    ys = [frac_results[f][metric] for f in fractions]
    ax.plot(xs_pct, ys, "o-", color="#4C72B0", linewidth=2, markersize=7)
    for x_pt, y_pt in zip(xs_pct, ys):
        ax.annotate(f"{y_pt:.3f}", (x_pt, y_pt),
                    textcoords="offset points", xytext=(0, 9),
                    ha="center", fontsize=8)
    ax.set_xlabel("Label fraction (%)", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f"LoTSS classical_pure — {ylabel}", fontsize=12)
    ax.set_xticks(xs_pct)
    ax.set_xticklabels([f"{x:.0f}%" for x in xs_pct])
    ax.set_ylim(0, 1.1)
    ax.grid(True, alpha=0.3)

plt.suptitle("Label Fraction vs Performance (LoTSS classical_pure, test split)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/label_fraction_lotss.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {OUT_DIR}/label_fraction_lotss.png")


## 7. Cross-Dataset Generalisation

Train a logistic regression on **LoTSS classical_pure train-split** projections,
then evaluate zero-shot on:
- **MiraBest test split** (binary FRI/FRII)
- **RadioGalaxyDataset test split** — FRI/FRII only (labels 0 and 1)


In [ ]:
_ensure("lotss", tag="classical_pure")
_ensure("mirabest")
_ensure("radio_galaxy_dataset")

# Fit on LoTSS classical_pure train
cp        = projs_cache["lotss_classical_pure"]
X_src     = cp["projs"][cp["split_ids"] == 0]
y_src     = cp["labels"][cp["split_ids"] == 0]
scaler_x  = StandardScaler().fit(X_src)
clf_cross = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
clf_cross.fit(scaler_x.transform(X_src), y_src)
print(f"Trained on {len(y_src)} LoTSS classical_pure train samples")

cross_results = {}

# Evaluate on MiraBest test
mb      = projs_cache["mirabest"]
mask_mb = mb["split_ids"] == 2
X_mb    = scaler_x.transform(mb["projs"][mask_mb])
y_mb    = mb["labels"][mask_mb]
yp_mb   = clf_cross.predict(X_mb)
ypr_mb  = clf_cross.predict_proba(X_mb)
cross_results["MiraBest"] = {
    "accuracy": accuracy_score(y_mb, yp_mb),
    "f1":       f1_score(y_mb, yp_mb, average="macro", zero_division=0),
    "auc":      roc_auc_score(y_mb, ypr_mb[:, 1]),
    "n":        int(mask_mb.sum()),
}

# Evaluate on RadioGalaxyDataset test — FRI/FRII only
rgd      = projs_cache["radio_galaxy_dataset"]
mask_rgd = (rgd["split_ids"] == 2) & ((rgd["labels"] == 0) | (rgd["labels"] == 1))
X_rgd    = scaler_x.transform(rgd["projs"][mask_rgd])
y_rgd    = rgd["labels"][mask_rgd]
yp_rgd   = clf_cross.predict(X_rgd)
ypr_rgd  = clf_cross.predict_proba(X_rgd)
cross_results["RadioGalaxyDataset (FRI/FRII)"] = {
    "accuracy": accuracy_score(y_rgd, yp_rgd),
    "f1":       f1_score(y_rgd, yp_rgd, average="macro", zero_division=0),
    "auc":      roc_auc_score(y_rgd, ypr_rgd[:, 1]),
    "n":        int(mask_rgd.sum()),
}

print("\nCross-dataset generalisation (trained on LoTSS classical_pure):")
rows = [{"target": k, "n_test": v["n"],
         "accuracy": round(v["accuracy"], 3),
         "macro_F1": round(v["f1"], 3),
         "macro_AUC": round(v["auc"], 3)}
        for k, v in cross_results.items()]
display(pd.DataFrame(rows).set_index("target"))


## 8. Summary Table

Final comparison across all datasets and evaluation methods.
**Bold** marks the better result per metric per dataset (linear probe vs fine-tune head).

In [ ]:
from IPython.display import HTML

datasets_order = [
    ("MiraBest",             "mirabest",             2),
    ("RadioGalaxyDataset",   "radio_galaxy_dataset", 4),
    ("LoTSS classical_pure", "lotss_classical_pure", 2),
]
metrics_order = ["accuracy", "f1", "auc"]

rows_html = []
for disp_name, key, _ in datasets_order:
    lp = probe_results.get(key, {}).get("test", {})
    ft = ft_results.get(key, {})
    vals = [lp.get(m, float("nan")) for m in metrics_order] +                [ft.get(m, float("nan")) for m in metrics_order]

    bold = [False] * 6
    for mi in range(3):
        lv, fv = vals[mi], vals[mi + 3]
        if not (lv != lv or fv != fv):   # skip if nan
            if lv >= fv: bold[mi]     = True
            else:        bold[mi + 3] = True

    tds = "".join(
        (f"<td><b>{v:.3f}</b></td>" if b else f"<td>{v:.3f}</td>")
        if v == v else "<td>—</td>"
        for v, b in zip(vals, bold)
    )
    rows_html.append(f"<tr><td><b>{disp_name}</b></td>{tds}</tr>")

hdr2 = "".join(f"<th>{h}</th>"
               for h in ["Acc","F1","AUC","Acc","F1","AUC"])
table = (
    "<table border=\"1\" style=\"border-collapse:collapse;font-family:monospace\">"
    "<thead>"
    "<tr><th rowspan=\"2\">Dataset</th>"
    "<th colspan=\"3\">Linear Probe (projections)</th>"
    "<th colspan=\"3\">Fine-tune Head (encodings)</th></tr>"
    f"<tr>{hdr2}</tr>"
    "</thead>"
    f"<tbody>{''.join(rows_html)}</tbody>"
    "</table>"
)
display(HTML(table))


In [ ]:
# Plain-text version
print(f"{'Dataset':<28} {'LP Acc':>7} {'LP F1':>6} {'LP AUC':>7}  "
      f"{'FT Acc':>7} {'FT F1':>6} {'FT AUC':>7}")
print("-" * 80)
for disp_name, key, _ in datasets_order:
    lp = probe_results.get(key, {}).get("test", {})
    ft = ft_results.get(key, {})
    def fmt(d, k):
        v = d.get(k, float("nan"))
        return f"{v:.3f}" if v == v else "  —  "
    print(f"{disp_name:<28} {fmt(lp,'accuracy'):>7} {fmt(lp,'f1'):>6} {fmt(lp,'auc'):>7}  "
          f"{fmt(ft,'accuracy'):>7} {fmt(ft,'f1'):>6} {fmt(ft,'auc'):>7}")
